In [17]:
# ========== CONFIGURATION ==========
# This cell sets up device (CUDA/CPU) and asks for paths

import torch
import os
from tkinter import filedialog
import tkinter as tk

# --- Check for CUDA and set device ---
if torch.cuda.is_available():
    device = "cuda"
    print("CUDA available. Using GPU for inference.")
    print("  GPU: " + torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("CUDA not available. Falling back to CPU.")
    print("  Note: Detection will be slower on CPU.")

# --- Get model weights path ---
print("--- Select Model Weights File ---")
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

model_weights_path = filedialog.askopenfilename(
    title="Select Model Weights (.pth file)",
    filetypes=[("PyTorch Model", "*.pth"), ("All Files", "*.*")],
    initialdir=os.getcwd()
)

if not model_weights_path:
    raise ValueError("No model weights selected. Please run this cell again.")

print("Selected weights: " + model_weights_path)

# --- Get image folder ---
print("--- Select Image Folder ---")
root2 = tk.Tk()
root2.withdraw()
root2.attributes('-topmost', True)

image_folder = filedialog.askdirectory(
    title="Select Folder Containing Images",
    initialdir=os.getcwd()
)

if not image_folder:
    raise ValueError("No folder selected. Please run this cell again.")

print("Selected folder: " + image_folder)


print("=" * 60)
print("Configuration complete")
print("Device:               " + device)
print("Weights:              " + os.path.basename(model_weights_path))
print("Image folder:         " + image_folder)
print("=" * 60)


CUDA not available. Falling back to CPU.
  Note: Detection will be slower on CPU.
--- Select Model Weights File ---
Selected weights: C:/Users/tkuak/Desktop/DropletID/model_1class.pth
--- Select Image Folder ---
Selected folder: C:/Users/tkuak/Desktop/DropletID/droplet_images/500x500px thrsh = var/3rd attempt
Configuration complete
Device:               cpu
Weights:              model_1class.pth
Image folder:         C:/Users/tkuak/Desktop/DropletID/droplet_images/500x500px thrsh = var/3rd attempt


In [18]:
# ------------- LOADING MODEL ----------------------
import cv2
import os
import matplotlib.pyplot as plt
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2 import model_zoo
import numpy as np
import torch
from detectron2.data import MetadataCatalog, DatasetCatalog
from scipy.ndimage import distance_transform_edt, maximum_filter, label as ndi_label
from skimage.measure import regionprops

In [19]:
from skimage.segmentation import watershed
import glob

# Use device from configuration cell
cfg = get_cfg()
cfg.OUTPUT_DIR = "./"
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1000
cfg.SOLVER.STEPS = []
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.WEIGHTS = model_weights_path
cfg.MODEL.DEVICE = device

thresh_coarse = 0.4
thresh_fine = 0.5

cfg_coarse = cfg.clone()
cfg_coarse.MODEL.ROI_HEADS.SCORE_THRESH_TEST = thresh_coarse
cfg_fine = cfg.clone()
cfg_fine.MODEL.ROI_HEADS.SCORE_THRESH_TEST = thresh_fine

print("Loading model on " + device + "...")
predictor_coarse = DefaultPredictor(cfg_coarse)
predictor_fine = DefaultPredictor(cfg_fine)
print("Model loaded successfully")

# -------------- DETECTION HELPER FUNCTION --------------
def run_detection_pass(image, predictor, rows, cols, shift_h=False, shift_w=False, max_mask_fraction=1/3, min_radius=0, max_radius=0):
    crop_h = image.shape[0] // rows
    crop_w = image.shape[1] // cols
    h_offset = crop_h // 2 if shift_h else 0
    w_offset = crop_w // 2 if shift_w else 0
    actual_rows = rows - 1 if shift_h else rows
    actual_cols = cols - 1 if shift_w else cols

    fullimg = np.empty((actual_rows, actual_cols), dtype=object)

    for k in range(actual_rows):
        for j in range(actual_cols):
            tile = image[
                k*crop_h + h_offset : k*crop_h + crop_h + h_offset,
                j*crop_w + w_offset : j*crop_w + crop_w + w_offset
            ]
            outputs = predictor(tile)
            masks = outputs['instances'].pred_masks.cpu()
            bb = outputs['instances'].pred_boxes

            # Edge filter
            for l in range(len(bb)-1, -1, -1):
                if bb[l].tensor.cpu().tolist()[0][0] < 10:
                    masks = torch.cat((masks[:l], masks[l+1:]), dim=0)
                elif bb[l].tensor.cpu().tolist()[0][1] < 10:
                    masks = torch.cat((masks[:l], masks[l+1:]), dim=0)
                elif bb[l].tensor.cpu().tolist()[0][2] > crop_w - 10:
                    masks = torch.cat((masks[:l], masks[l+1:]), dim=0)
                elif bb[l].tensor.cpu().tolist()[0][3] > crop_h - 10:
                    masks = torch.cat((masks[:l], masks[l+1:]), dim=0)

            # Min radius filter
            if min_radius > 0:
                min_area = np.pi * (min_radius ** 2)
                for l in range(len(masks)-1, -1, -1):
                    if masks[l].sum().item() < min_area:
                        masks = torch.cat((masks[:l], masks[l+1:]), dim=0)

            # Max radius filter
            if max_radius > 0:
                max_area = np.pi * (max_radius ** 2)
                for l in range(len(masks)-1, -1, -1):
                    if masks[l].sum().item() > max_area:
                        masks = torch.cat((masks[:l], masks[l+1:]), dim=0)

            # Max size filter
            if len(masks) > 0:
                combined_mask = torch.zeros_like(masks[0], dtype=torch.bool)
                for i in range(len(masks)):
                    if masks[i].sum().item() < int(crop_h * crop_w * max_mask_fraction):
                        combined_mask |= masks[i]
            else:
                combined_mask = torch.zeros((crop_h, crop_w), dtype=torch.bool)

            fullimg[k, j] = combined_mask

    img_rows_list = [np.concatenate([fullimg[i, j] for j in range(actual_cols)], axis=1) for i in range(actual_rows)]
    result = np.concatenate(img_rows_list, axis=0)

    if shift_h:
        add_h = 1 if crop_h % 2 == 1 else 0
        top_pad = np.zeros((h_offset, result.shape[1]), dtype=bool)
        bottom_pad = np.zeros((h_offset + image.shape[0] - crop_h * rows + add_h, result.shape[1]), dtype=bool)
        result = np.concatenate([top_pad, result.astype(bool), bottom_pad], axis=0)
    else:
        bottom_pad = np.zeros((image.shape[0] - crop_h * rows, result.shape[1]), dtype=bool)
        result = np.concatenate([result.astype(bool), bottom_pad], axis=0)

    if shift_w:
        add_w = 1 if crop_w % 2 == 1 else 0
        left_pad = np.zeros((result.shape[0], w_offset), dtype=bool)
        right_pad = np.zeros((result.shape[0], w_offset + image.shape[1] - crop_w * cols + add_w), dtype=bool)
        result = np.concatenate([left_pad, result, right_pad], axis=1)
    else:
        right_pad = np.zeros((result.shape[0], image.shape[1] - crop_w * cols), dtype=bool)
        result = np.concatenate([result, right_pad], axis=1)

    return result

# -------------- GRID VISUALIZATION HELPER --------------
def save_grid_image(image, rows, cols, shift_h, shift_w, pass_num, output_dir, base_name):
    crop_h = image.shape[0] // rows
    crop_w = image.shape[1] // cols
    h_offset = crop_h // 2 if shift_h else 0
    w_offset = crop_w // 2 if shift_w else 0
    actual_rows = rows - 1 if shift_h else rows
    actual_cols = cols - 1 if shift_w else cols

    vis = image.copy()

    for k in range(actual_rows + 1):
        y = h_offset + k * crop_h
        cv2.line(vis, (0, y), (image.shape[1] - 1, y), (0, 255, 0), 2)

    for j in range(actual_cols + 1):
        x = w_offset + j * crop_w
        cv2.line(vis, (x, 0), (x, image.shape[0] - 1), (0, 255, 0), 2)

    out_path = os.path.join(output_dir, base_name + "_grid_pass" + str(pass_num) + ".png")
    cv2.imwrite(out_path, vis)
    return out_path

# -------------- WATERSHED + REGIONS HELPER --------------
def get_regions(mask):
    mask_uint8 = (mask > 0).astype(np.uint8) * 255
    contour_data = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    clusters = contour_data[0] if len(contour_data) == 2 else contour_data[1]
    split_label_mask = np.zeros_like(mask_uint8, dtype=np.int32)
    current_label = 1
    for cluster in clusters:
        cluster_mask = np.zeros_like(mask_uint8)
        cv2.drawContours(cluster_mask, [cluster], -1, 255, -1)
        cluster_bool = cluster_mask > 0
        cluster_dist = distance_transform_edt(cluster_bool)
        local_max = (cluster_dist == maximum_filter(cluster_dist, size=40)) & (cluster_dist > 0)
        markers_labeled, n_markers = ndi_label(local_max.astype(np.uint8))
        if n_markers <= 1:
            split_label_mask[cluster_bool] = current_label
            current_label += 1
            continue
        labels = watershed(-cluster_dist, markers=markers_labeled, mask=cluster_bool)
        for sublabel in range(1, labels.max() + 1):
            split_label_mask[(labels == sublabel)] = current_label
            current_label += 1
    regions = regionprops(split_label_mask)
    overlap_threshold = 0.5
    sorted_regions = sorted(regions, key=lambda r: r.area, reverse=True)
    kept = []
    for region in sorted_regions:
        ry, rx = region.centroid
        rr = region.equivalent_diameter / 2
        overlaps = False
        for kept_r in kept:
            ky, kx = kept_r.centroid
            kr = kept_r.equivalent_diameter / 2
            center_dist = np.sqrt((ry - ky)**2 + (rx - kx)**2)
            if center_dist < (rr + kr) * overlap_threshold:
                overlaps = True
                break
        if not overlaps:
            kept.append(region)
    return kept

# -------------- CIRCLES IMAGE HELPER --------------
def save_circles_image(image, regions, path):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    for region in regions:
        y, x = region.centroid
        ediam = region.equivalent_diameter
        circ = plt.Circle((x, y), ediam / 2, edgecolor="red", facecolor="none", linewidth=2)
        ax.add_patch(circ)
    ax.set_axis_off()
    fig.tight_layout()
    plt.savefig(path, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

# -------------- GRID SCALES --------------
rows1, cols1 = 2, 2
rows2, cols2 = 5, 5

# pass1: rows1 x cols1 grid
# pass2: 1 x 2  (vertical split at center)
# pass3: 2 x 1  (horizontal split at center)
# pass4-7: rows2 x cols2 grid with shifts
grid_params = [
    (rows1, cols1, False, False, 1),
    (1,     2,     False, False, 2),
    (2,     1,     False, False, 3),
    (rows2, cols2, False, False, 4),
    (rows2, cols2, False, True,  5),
    (rows2, cols2, True,  False, 6),
    (rows2, cols2, True,  True,  7),
]

# -------------- FIND ALL IMAGES IN FOLDER --------------
extensions = ['*.tif', '*.tiff', '*.png', '*.jpg', '*.jpeg']
image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(image_folder, ext)))
image_paths.sort()

if not image_paths:
    raise ValueError("No images found in folder: " + image_folder)

print("Found " + str(len(image_paths)) + " images to process.")

# -------------- PROCESS EACH IMAGE --------------
for img_idx, image_path in enumerate(image_paths):
    is_first = (img_idx == 0)
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    input_dir = os.path.dirname(image_path)

    print("=" * 60)
    print("[" + str(img_idx+1) + "/" + str(len(image_paths)) + "] " + base_name)

    image = cv2.imread(image_path)
    if image is None:
        print("  WARNING: Could not load image, skipping.")
        continue

    print("  Shape: " + str(image.shape))

    # Minimum radius for large-tile passes = 1/20 of image height
    min_r = image.shape[0] / 50

    # --- Detection passes (each pass masks out previously detected regions) ---
    cumulative_mask = np.zeros(image.shape[:2], dtype=bool)

    print("  Pass 1: " + str(rows1) + "x" + str(cols1) + " grid...")
    pass1 = run_detection_pass(image, predictor_coarse, rows1, cols1, max_mask_fraction=0.75, min_radius=min_r)
    cumulative_mask |= pass1

    print("  Pass 2: vertical split (1x2)...")
    img2 = image.copy(); img2[cumulative_mask] = 0
    pass2 = run_detection_pass(img2, predictor_coarse, 1, 2, max_mask_fraction=0.75, min_radius=min_r)
    cumulative_mask |= pass2

    print("  Pass 3: horizontal split (2x1)...")
    img3 = image.copy(); img3[cumulative_mask] = 0
    pass3 = run_detection_pass(img3, predictor_coarse, 2, 1, max_mask_fraction=0.75, min_radius=min_r)
    cumulative_mask |= pass3

    print("  Pass 4: " + str(rows2) + "x" + str(cols2) + " grid...")
    img4 = image.copy(); img4[cumulative_mask] = 0
    pass4 = run_detection_pass(img4, predictor_fine, rows2, cols2, max_radius=min_r)
    cumulative_mask |= pass4

    print("  Pass 5: " + str(rows2) + "x" + str(cols2) + " grid, shift right...")
    img5 = image.copy(); img5[cumulative_mask] = 0
    pass5 = run_detection_pass(img5, predictor_fine, rows2, cols2, shift_w=True, max_radius=min_r)
    cumulative_mask |= pass5

    print("  Pass 6: " + str(rows2) + "x" + str(cols2) + " grid, shift down...")
    img6 = image.copy(); img6[cumulative_mask] = 0
    pass6 = run_detection_pass(img6, predictor_fine, rows2, cols2, shift_h=True, max_radius=min_r)
    cumulative_mask |= pass6

    print("  Pass 7: " + str(rows2) + "x" + str(cols2) + " grid, shift both...")
    img7 = image.copy(); img7[cumulative_mask] = 0
    pass7 = run_detection_pass(img7, predictor_fine, rows2, cols2, shift_h=True, shift_w=True, max_radius=min_r)

    # --- Grid visualization (first image only) ---
    if is_first:
        print("  Saving grid visualizations...")
        for grows, gcols, gsh, gsw, gpnum in grid_params:
            gpath = save_grid_image(image, grows, gcols, gsh, gsw, gpnum, input_dir, base_name)
            print("    Pass " + str(gpnum) + ": " + os.path.basename(gpath))

    # --- Per-pass circle images ---
    print("  Saving per-pass circle images...")
    for pass_num, pass_mask in enumerate([pass1, pass2, pass3, pass4, pass5, pass6, pass7], start=1):
        pass_regions = get_regions(pass_mask)
        pass_circles_path = os.path.join(input_dir, base_name + "_circles_pass" + str(pass_num) + ".png")
        save_circles_image(image, pass_regions, pass_circles_path)
        print("    Pass " + str(pass_num) + ": " + str(len(pass_regions)) + " droplets -> " + os.path.basename(pass_circles_path))

    # --- Combine passes ---
    full_mask = pass1 | pass2 | pass3 | pass4 | pass5 | pass6 | pass7

    # --- Final regions ---
    regions = get_regions(full_mask)
    diameters = [region.equivalent_diameter for region in regions]
    print("  Detected " + str(len(regions)) + " droplets (combined)")

    # --- Save diameters text file ---
    diameter_txt_path = os.path.join(input_dir, base_name + "_diameters.txt")
    np.savetxt(diameter_txt_path, diameters)

    # --- Save circles on original image ---
    circles_orig_path = os.path.join(input_dir, base_name + "_circles_original.png")
    save_circles_image(image, regions, circles_orig_path)

    print("  Saved: " + os.path.basename(diameter_txt_path))
    print("  Saved: " + os.path.basename(circles_orig_path))

print("=" * 60)
print("ALL IMAGES PROCESSED")
print("=" * 60)


Loading model on cpu...


c:\Users\tkuak\Desktop\DropletID\venv\lib\site-packages\fvcore\common\checkpoint.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=torc

Model loaded successfully
Found 1 images to process.
[1/1] 133_crop
  Shape: (500, 500, 3)
  Pass 1: 2x2 grid...
  Pass 2: vertical split (1x2)...
  Pass 3: horizontal split (2x1)...
  Pass 4: 5x5 grid...
  Pass 5: 5x5 grid, shift right...
  Pass 6: 5x5 grid, shift down...
  Pass 7: 5x5 grid, shift both...
  Saving grid visualizations...
    Pass 1: 133_crop_grid_pass1.png
    Pass 2: 133_crop_grid_pass2.png
    Pass 3: 133_crop_grid_pass3.png
    Pass 4: 133_crop_grid_pass4.png
    Pass 5: 133_crop_grid_pass5.png
    Pass 6: 133_crop_grid_pass6.png
    Pass 7: 133_crop_grid_pass7.png
  Saving per-pass circle images...
    Pass 1: 38 droplets -> 133_crop_circles_pass1.png
    Pass 2: 42 droplets -> 133_crop_circles_pass2.png
    Pass 3: 36 droplets -> 133_crop_circles_pass3.png
    Pass 4: 384 droplets -> 133_crop_circles_pass4.png
    Pass 5: 267 droplets -> 133_crop_circles_pass5.png
    Pass 6: 204 droplets -> 133_crop_circles_pass6.png
    Pass 7: 139 droplets -> 133_crop_circles_p

In [20]:
# Check if cropping was done - skip FOV processing if not
if x_min is None or y_min is None:
    print("Skipping FOV processing - no crop was applied to the image.")
    print("To enable FOV processing, set x_min, y_min, x_max, y_max values in the detection cell.")
else:
    import numpy as np
    import cv2
    import tifffile
    import os
    import matplotlib.pyplot as plt

    # --- USER-SPECIFIED FOV from original image ---
    fov_xmin_orig, fov_xmax_orig = 839, 955
    fov_ymin_orig, fov_ymax_orig = 1460, 1579

    # --- CROP ORIGIN from cell one ---
    # x_min, y_min set above as 262, 614 -- do NOT change these here!

    # --- Convert to cropped image coordinates ---
    fov_xmin = fov_xmin_orig - x_min
    fov_xmax = fov_xmax_orig - x_min
    fov_ymin = fov_ymin_orig - y_min
    fov_ymax = fov_ymax_orig - y_min

    print(f"Working with FOV on cropped image: x:{fov_xmin}:{fov_xmax}, y:{fov_ymin}:{fov_ymax}")
    print(f"Image shape: {image.shape}")

    # --- Check bounds ---
    H, W = image.shape[:2]
    if not (0 <= fov_xmin < fov_xmax <= W) or not (0 <= fov_ymin < fov_ymax <= H):
        raise ValueError("FOV crop indices are out of bounds! Please check input.")

    # --- Now crop your FOV in image and mask, then proceed as before ---
    fov_img = image[fov_ymin:fov_ymax, fov_xmin:fov_xmax]
    # etc: use fov_img, and the same slices for masks/overlays as in prior cell

    # ========== 1. Crop the image in the FOV region ==========
    fov_img = image[fov_ymin:fov_ymax, fov_xmin:fov_xmax]
    fov_img_path = os.path.join(input_dir, f"{base_name}_FOV_image.tif")
    tifffile.imwrite(fov_img_path, fov_img)  # Use tifffile to preserve depth/channels

    # ========== 2. Find regions whose centroid is inside the FOV ==========
    fov_regions = []
    fov_indices = []
    fov_mask = np.zeros_like(full_mask, dtype=bool)
    for idx, region in enumerate(regions):
        y, x = region.centroid
        if fov_ymin <= y < fov_ymax and fov_xmin <= x < fov_xmax:
            fov_regions.append(region)
            fov_indices.append(idx)
            fov_mask[full_mask == region.label] = True

    # Crop the FOV mask to the FOV region
    cropped_fov_mask = fov_mask[fov_ymin:fov_ymax, fov_xmin:fov_xmax]

    # Color FOV mask: purple background, yellow objects
    mask_rgb_fov = np.zeros((*cropped_fov_mask.shape, 3), dtype=np.uint8)
    mask_rgb_fov[cropped_fov_mask == 0] = [128, 0, 128]
    mask_rgb_fov[cropped_fov_mask == 1] = [255, 255, 0]
    fov_mask_path = os.path.join(input_dir, f"{base_name}_FOV_mask.png")
    cv2.imwrite(fov_mask_path, cv2.cvtColor(mask_rgb_fov, cv2.COLOR_RGB2BGR))

    # ========== 3. FOV circles overlay ==========
    fov_circles_path = os.path.join(input_dir, f"{base_name}_FOV_circles.png")
    fig, ax = plt.subplots(figsize=(8,8))
    ax.imshow(cropped_fov_mask, cmap="gray", alpha=0.3)
    ax.imshow(fov_img, alpha=0.8)
    for region in fov_regions:
        y, x = region.centroid
        ediam = region.equivalent_diameter
        # Draw only if centroid within cropped FOV
        circ = plt.Circle((x-fov_xmin, y-fov_ymin), ediam/2, edgecolor='red', facecolor='none', linewidth=2)
        ax.add_patch(circ)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(fov_circles_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    # ========== 4. Save FOV diameters ==========
    fov_diameters = np.array([region.equivalent_diameter for region in fov_regions])
    fov_diam_path = os.path.join(input_dir, f"{base_name}_FOV_diameters.txt")
    np.savetxt(fov_diam_path, fov_diameters)

    print(f"Saved FOV cropped image (tif): {fov_img_path}")
    print(f"Saved FOV mask (png): {fov_mask_path}")
    print(f"Saved FOV overlay with fitted circles (png): {fov_circles_path}")
    print(f"Saved FOV droplet diameters (txt): {fov_diam_path}")

NameError: name 'x_min' is not defined